In [2]:
import glob, re, datetime
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

%matplotlib inline

In [1]:
cd ../..

/home/users/cornkle/pythonWorkspace/proj_CEH


In [30]:
path = '/home/users/cornkle/linked_CP4'
path_hist = path + '/hist/'
path_fut = path + '/future/'

In [31]:
# ---------------------------------------------------------------------
# User settings
# ---------------------------------------------------------------------

SM_VAR = "e08223"
RAIN_VAR = "a04203"

hist_sm_dir = path_hist + "/SM"
hist_rain_dir = path_hist + "/lsRain" 
fut_sm_dir  = path_fut + "/SM" 
fut_rain_dir  = path_fut + "/lsRain" 
# Start with a small test box. Set to None later for whole domain.
box = dict(latitude=slice(8, 16), longitude=slice(-12, 5))

# Test period using filename dates, not decoded cftime.
date0 = "19981201"
date1 = "19990115"

# Soil layer: 0 = top layer. You can also sum layers later.
depth_index = 0

# Rain threshold in mm h-1 after converting from kg m-2 s-1.
rain_thr = 0.1

# Maximum drydown lag to composite, in hours.
max_lag = 120

# For quick maps later: approximate 1 degree from 4.5 km grid.
# 1 degree / 0.04-ish degree ≈ 25 grid cells.
coarsen_xy = 25

In [35]:
import re
import os

def file_start_yyyymmdd(path):
    name = os.path.basename(path)
    m = re.search(r"_(\d{12})-\d{12}", name)
    return m.group(1)[:8] if m else None


def file_start_month(path):
    d = file_start_yyyymmdd(path)
    return int(d[4:6]) if d is not None else None


def filter_files(paths, date0=None, date1=None, months=None):
    paths = sorted(paths)

    if months is not None:
        months = set(months)

    out = []
    for p in paths:
        d = file_start_yyyymmdd(p)
        m = file_start_month(p)

        if d is None:
            continue

        if date0 is not None and d < date0:
            continue
        if date1 is not None and d > date1:
            continue
        if months is not None and m not in months:
            continue

        out.append(p)

    print(f"Kept {len(out)} files")
    if len(out) > 0:
        print("First kept:", os.path.basename(out[0]))
        print("Last kept: ", os.path.basename(out[-1]))

    return out


def shift_rain_time_to_hour_end(da):
    # Rain files are means at xx:30. Shift by +30 min to match SM at xx:00.
    new_time = [t + datetime.timedelta(minutes=30) if getattr(t, "minute", 0) == 30 else t for t in da.time.values]
    return da.assign_coords(time=new_time)



def open_cp4_sm(sm_dir, date0=None, date1=None, months=None, box=None, depth_index=0, chunks=None):
    all_files = sorted(glob.glob(sm_dir + "/*.nc"))
    files = filter_files(all_files, date0=date0, date1=date1, months=months)

    if len(files) == 0:
        raise FileNotFoundError(f"No SM files found in {sm_dir}")

    def pp(ds):
        da = ds[SM_VAR].isel(depth=depth_index)
        if box is not None:
            da = da.sel(**box)
        return da.to_dataset(name="SM")

    if chunks is None:
        chunks = {"time": 24}

    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_mfdataset(files, combine="by_coords", preprocess=pp, chunks=chunks, decode_times=time_coder)
    return ds["SM"].sortby("time")


def open_cp4_rain(rain_dir, date0=None, date1=None, months=None, box=None, chunks=None):
    all_files = sorted(glob.glob(rain_dir + "/*.nc"))
    files = filter_files(all_files, date0=date0, date1=date1, months=months)

    if len(files) == 0:
        raise FileNotFoundError(f"No rain files found in {rain_dir}")

    def pp(ds):
        da = ds[RAIN_VAR] * 3600.0
        da.attrs["units"] = "mm h-1"
        if box is not None:
            da = da.sel(**box)
        return da.to_dataset(name="rain")

    if chunks is None:
        chunks = {"time": 24}

    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_mfdataset(files, combine="by_coords", preprocess=pp, chunks=chunks, decode_times=time_coder)

    rain = shift_rain_time_to_hour_end(ds["rain"].sortby("time"))
    return rain

In [36]:
def drydown_composite(sm, rain, rain_thr=0.1, max_lag=120, min_start_rain=0.1):
    # Align rain to SM times.
    rain = rain.reindex(time=sm.time, method="nearest")

    wet = rain > rain_thr
    dry = ~wet

    # Start of a drydown = dry now, wet previous hour.
    start = dry & wet.shift(time=1, fill_value=False)

    lags = np.arange(max_lag + 1)
    curves, valid_counts = [], []
    spell_lengths = []

    for lag in lags:
        sm_lag = sm.shift(time=-lag)
        dry_lag = dry.shift(time=-lag, fill_value=False)

        # Require all intervening hours to be dry up to this lag.
        if lag == 0:
            valid = start
        else:
            valid = start.copy()
            for j in range(1, lag + 1):
                valid = valid & dry.shift(time=-j, fill_value=False)

        sm0 = sm.where(start)
        dsm = sm_lag - sm0
        curves.append(dsm.where(valid).mean(("time", "latitude", "longitude")))
        valid_counts.append(valid.sum(("time", "latitude", "longitude")))

    drydown_curve = xr.concat(curves, dim=xr.DataArray(lags, dims="lag", name="lag"))
    valid_counts = xr.concat(valid_counts, dim=xr.DataArray(lags, dims="lag", name="lag"))

    # Dry-spell length after each drydown start.
    # For memory safety this computes over the already subsetted domain.
    wet_np = wet.load().values
    start_np = start.load().values

    nt, ny, nx = wet_np.shape
    spell = np.full((nt, ny, nx), np.nan, dtype=np.float32)

    for t in range(nt):
        active = start_np[t]
        if not active.any():
            continue

        max_possible = min(max_lag, nt - t - 1)
        length = np.zeros((ny, nx), dtype=np.float32)

        for j in range(1, max_possible + 1):
            still_dry = ~wet_np[t + j]
            length[active & still_dry] = j
            active = active & still_dry
            if not active.any():
                break

        spell[t] = np.where(start_np[t], length, np.nan)

    mean_dry_spell_length = np.nanmean(spell)

    # Drydown slope from lag 0..max_lag, weighted only where enough samples exist.
    y = drydown_curve.values
    ok = np.isfinite(y) & (valid_counts.values > 100)
    if ok.sum() >= 4:
        drydown_slope = np.polyfit(lags[ok], y[ok], 1)[0]
    else:
        drydown_slope = np.nan

    out = xr.Dataset({
        "drydown_curve": drydown_curve,
        "valid_counts": valid_counts,
        "mean_dry_spell_length": mean_dry_spell_length,
        "drydown_slope": drydown_slope,
        "n_events": start.sum(("time", "latitude", "longitude"))
    })

    return out

In [37]:
months = [7, 8, 9]

sm_h = open_cp4_sm(hist_sm_dir, date0=None, date1=None, months=months, box=box, depth_index=depth_index)
rr_h = open_cp4_rain(hist_rain_dir, date0=None, date1=None, months=months, box=box)

sm_f = open_cp4_sm(fut_sm_dir, date0=None, date1=None, months=months, box=box, depth_index=depth_index)
rr_f = open_cp4_rain(fut_rain_dir, date0=None, date1=None, months=months, box=box)

Kept 900 files
First kept: e08223_A1hr_inst_ad251_4km_199807010100-199807020000.nc
Last kept:  e08223_A1hr_inst_aj575_4km_200609300100-200610010000.nc


/tmp/ipykernel_1741/3354139267.py:70: FutureWarning: In a future version of xarray the default value for coords will change from coords='different' to coords='minimal'. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set coords explicitly.
  ds = xr.open_mfdataset(files, combine="by_coords", preprocess=pp, chunks=chunks, decode_times=time_coder)


Kept 900 files
First kept: a04203_A1hr_mean_ad251_4km_199807010030-199807012330.nc
Last kept:  a04203_A1hr_mean_aj575_4km_200609300030-200609302330.nc


/tmp/ipykernel_1741/3354139267.py:92: FutureWarning: In a future version of xarray the default value for coords will change from coords='different' to coords='minimal'. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set coords explicitly.
  ds = xr.open_mfdataset(files, combine="by_coords", preprocess=pp, chunks=chunks, decode_times=time_coder)


Kept 899 files
First kept: e08223_A1hr_mean_an298_4km_200207010030-200207012330.nc
Last kept:  e08223_A1hr_mean_aq679_4km_200109300030-200109302330.nc
Kept 900 files
First kept: a04203_A1hr_mean_an298_fc4km_200207010030-200207012330.nc
Last kept:  a04203_A1hr_mean_aq679_fc4km_200109300030-200109302330.nc


In [ ]:
comp_h = drydown_composite(sm_h, rr_h, rain_thr=rain_thr, max_lag=max_lag).compute()
comp_f = drydown_composite(sm_f, rr_f, rain_thr=rain_thr, max_lag=max_lag).compute()

/tmp/ipykernel_1741/1854816505.py:60: RuntimeWarning: Mean of empty slice
  mean_dry_spell_length = np.nanmean(spell)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5), dpi=120)

ax.plot(comp_h.lag / 24, comp_h.drydown_curve, label="hist")
ax.plot(comp_f.lag / 24, comp_f.drydown_curve, label="future")

ax.axhline(0, lw=0.8, color="0.5")
ax.set_xlabel("Days since last rain")
ax.set_ylabel("Δ soil moisture since rain event (kg m$^{-2}$)")
ax.set_title("Post-rain soil moisture drydown")
ax.legend(frameon=False)

txt = (
    f"Hist: dry spell={float(comp_h.mean_dry_spell_length)/24:.1f} d, "
    f"slope={float(comp_h.drydown_slope)*24:.2f} kg m$^{{-2}}$ d$^{{-1}}$\n"
    f"Fut: dry spell={float(comp_f.mean_dry_spell_length)/24:.1f} d, "
    f"slope={float(comp_f.drydown_slope)*24:.2f} kg m$^{{-2}}$ d$^{{-1}}$"
)

ax.text(0.02, 0.04, txt, transform=ax.transAxes, va="bottom", ha="left", fontsize=9)
plt.tight_layout()

In [ ]:
def drydown_metrics_2d(sm, rain, rain_thr=0.1, max_lag=96, min_events=20):
    rain = rain.reindex(time=sm.time, method="nearest")
    wet = rain > rain_thr
    dry = ~wet
    start = dry & wet.shift(time=1, fill_value=False)

    # Mean dry-spell length per pixel.
    wet_np = wet.load().values
    start_np = start.load().values
    nt, ny, nx = wet_np.shape

    spell_sum = np.zeros((ny, nx), dtype=np.float64)
    spell_n = np.zeros((ny, nx), dtype=np.float64)

    for t in range(nt):
        active0 = start_np[t]
        if not active0.any():
            continue

        active = active0.copy()
        length = np.zeros((ny, nx), dtype=np.float32)
        max_possible = min(max_lag, nt - t - 1)

        for j in range(1, max_possible + 1):
            still_dry = ~wet_np[t + j]
            length[active & still_dry] = j
            active = active & still_dry
            if not active.any():
                break

        spell_sum += np.where(active0, length, 0)
        spell_n += active0

    mean_spell = xr.DataArray(spell_sum / np.where(spell_n == 0, np.nan, spell_n),
                              coords={"latitude": sm.latitude, "longitude": sm.longitude},
                              dims=("latitude", "longitude"))

    n_events = xr.DataArray(spell_n, coords=mean_spell.coords, dims=mean_spell.dims)

    # Drydown slope per pixel from composite lags.
    lags = np.arange(max_lag + 1)
    curves = []

    for lag in lags:
        sm_lag = sm.shift(time=-lag)
        if lag == 0:
            valid = start
        else:
            valid = start.copy()
            for j in range(1, lag + 1):
                valid = valid & dry.shift(time=-j, fill_value=False)

        sm0 = sm.where(start)
        curves.append((sm_lag - sm0).where(valid).mean("time"))

    curve = xr.concat(curves, dim=xr.DataArray(lags, dims="lag", name="lag"))

    # Fit slope over lag dimension. Slope is usually negative; drying rate k is positive.
    slope = curve.polyfit(dim="lag", deg=1)["polyfit_coefficients"].sel(degree=1)
    k = -slope

    mean_spell = mean_spell.where(n_events >= min_events)
    k = k.where(n_events >= min_events)

    return xr.Dataset({"dry_spell_h": mean_spell, "drying_rate_per_h": k, "n_events": n_events})

In [ ]:
box_waf = dict(latitude=slice(5, 20), longitude=slice(-18, 15))

sm_h = open_cp4_sm(hist_sm_dir, date0=date0, date1=date1, box=box_waf, depth_index=depth_index)
rr_h = open_cp4_rain(hist_rain_dir, date0=date0, date1=date1, box=box_waf)

sm_f = open_cp4_sm(fut_sm_dir, date0=date0, date1=date1, box=box_waf, depth_index=depth_index)
rr_f = open_cp4_rain(fut_rain_dir, date0=date0, date1=date1, box=box_waf)

# Coarsen to roughly 1 degree.
sm_h_1d = sm_h.coarsen(latitude=coarsen_xy, longitude=coarsen_xy, boundary="trim").mean()
rr_h_1d = rr_h.coarsen(latitude=coarsen_xy, longitude=coarsen_xy, boundary="trim").mean()

sm_f_1d = sm_f.coarsen(latitude=coarsen_xy, longitude=coarsen_xy, boundary="trim").mean()
rr_f_1d = rr_f.coarsen(latitude=coarsen_xy, longitude=coarsen_xy, boundary="trim").mean()

mh = drydown_metrics_2d(sm_h_1d, rr_h_1d, rain_thr=rain_thr, max_lag=96).compute()
mf = drydown_metrics_2d(sm_f_1d, rr_f_1d, rain_thr=rain_thr, max_lag=96).compute()

In [ ]:
Lh = mh["dry_spell_h"]
Lf = mf["dry_spell_h"]

kh = mh["drying_rate_per_h"]
kf = mf["drying_rate_per_h"]

dL = Lf - Lh
dk = kf - kh

storm_spacing_effect = kh * dL
drydown_rate_effect = Lh * dk

dominance = xr.where(abs(drydown_rate_effect) > abs(storm_spacing_effect), 1, -1)
dominance = dominance.where(np.isfinite(storm_spacing_effect + drydown_rate_effect))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), dpi=120, constrained_layout=True)

(storm_spacing_effect * 24).plot(ax=axes[0], cmap="RdBu_r", robust=True,
                                 cbar_kwargs={"label": "kg m$^{-2}$ d$^{-1}$ equivalent"})
axes[0].set_title("Drying from changed storm spacing")

(drydown_rate_effect * 24).plot(ax=axes[1], cmap="RdBu_r", robust=True,
                                cbar_kwargs={"label": "kg m$^{-2}$ d$^{-1}$ equivalent"})
axes[1].set_title("Drying from faster drydown")

dominance.plot(ax=axes[2], cmap="bwr", levels=[-1.5, 0, 1.5],
               cbar_kwargs={"ticks": [-1, 1], "label": "-1 spacing, +1 drydown"})
axes[2].set_title("Dominant contribution")

for ax in axes:
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")

plt.show()